## 1. Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from math import radians, sin, cos, sqrt, atan2

In [ ]:
lol

## 2. Load Data

In [2]:
# Navigate to your project
os.chdir('/workspaces/DSE3101-Project')

# Verify you're in the right place
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder
os.chdir('data/raw')


Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']


In [3]:
hdb_df = pd.read_csv('HDB_full_resale_info.csv.gz')
hdb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 261446 entries, 0 to 261445
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   town                     261446 non-null  str    
 1   flat_type                261446 non-null  str    
 2   block                    261446 non-null  str    
 3   street_name              261446 non-null  str    
 4   storey_range             261446 non-null  str    
 5   floor_area_sqm           261446 non-null  float64
 6   flat_model               261446 non-null  str    
 7   lease_commence_date      261446 non-null  int64  
 8   resale_price             261446 non-null  float64
 9   remaining_lease          261446 non-null  int64  
 10  sold_year                261446 non-null  int64  
 11  address                  261446 non-null  str    
 12  max_floor_lvl            261446 non-null  int64  
 13  storey_mid               261446 non-null  float64
 14  storey_category

In [4]:
final_duplicates = hdb_df.duplicated().sum()
print(f"Duplicate rows found: {final_duplicates}")

if final_duplicates > 0:
    hdb_df = hdb_df.drop_duplicates()

Duplicate rows found: 2303


In [5]:
# Merge RPI
rpi_df = pd.read_csv('HousingAndDevelopmentBoardHDBResalePriceIndex1Q2009100Quarterly.csv')
rpi_long = rpi_df.melt(id_vars='DataSeries', var_name='quarter_str', value_name='RPI')
rpi_long = rpi_long[rpi_long['DataSeries'] == 'HDB Resale Price Index'].copy()
rpi_long['sold_year'] = rpi_long['quarter_str'].str[:4].astype(int)
rpi_annual = rpi_long.groupby('sold_year')['RPI'].mean().reset_index()
hdb_df = hdb_df.merge(rpi_annual, on='sold_year', how='left')

hdb_df = hdb_df.dropna(subset=['RPI'])
print(f"After dropping missing RPI: {hdb_df.shape}")

print(f"Dataset shape: {hdb_df.shape}")
hdb_df.info()

After dropping missing RPI: (256457, 22)
Dataset shape: (256457, 22)
<class 'pandas.DataFrame'>
RangeIndex: 256457 entries, 0 to 256456
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   town                     256457 non-null  str    
 1   flat_type                256457 non-null  str    
 2   block                    256457 non-null  str    
 3   street_name              256457 non-null  str    
 4   storey_range             256457 non-null  str    
 5   floor_area_sqm           256457 non-null  float64
 6   flat_model               256457 non-null  str    
 7   lease_commence_date      256457 non-null  int64  
 8   resale_price             256457 non-null  float64
 9   remaining_lease          256457 non-null  int64  
 10  sold_year                256457 non-null  int64  
 11  address                  256457 non-null  str    
 12  max_floor_lvl            256457 non-null  int64  
 13  s

## 3. Feature Engineering

In [6]:
df_model = hdb_df.copy()

# --- Drop rows with missing RPI ---
df_model = df_model.dropna(subset=['RPI'])

# --- Flat age ---
df_model['flat_age_at_sale'] = df_model['sold_year'] - df_model['lease_commence_date']

# --- Normalized target ---
df_model['log_price_norm'] = np.log(df_model['resale_price'] / (df_model['floor_area_sqm'] * df_model['RPI']))

# --- Drop non-feature columns ---
cols_to_drop = ['address', 'block', 'street_name', 'RPI', 'resale_price']
df_model = df_model.drop(columns=[c for c in cols_to_drop if c in df_model.columns])

# --- Encode categoricals ---
categorical_cols = df_model.select_dtypes(include=['object', 'str']).columns.tolist()
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le

print(f"Preprocessed shape: {df_model.shape}")
print(f"Columns: {df_model.columns.tolist()}")

Preprocessed shape: (256457, 19)
Columns: ['town', 'flat_type', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'sold_year', 'max_floor_lvl', 'storey_mid', 'storey_category', 'is_mature_estate', 'eldercare_count_1km', 'clinic_count_1km', 'hospital_count_1km', 'communityclub_count_1km', 'park_count_1km', 'flat_age_at_sale', 'log_price_norm']


## 4. Train/Test Split

In [7]:
TREE_FEATURES = [c for c in df_model.columns if c not in ['log_price_norm']]

X_train, X_test, y_train, y_test = train_test_split(
    df_model[TREE_FEATURES],
    df_model['log_price_norm'],
    test_size=0.2,
    random_state=42
)

CURRENT_RPI = 203.6
y_test_actual = np.exp(y_test) * X_test['floor_area_sqm'] * CURRENT_RPI

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")

Training set: 205,165 rows
Test set:     51,292 rows


In [8]:
sample_weights = np.where(X_train['sold_year'] >= 2022, 2.0, 1.0)

xgb_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
print("Training complete.")

Training complete.


## 5. Evaluation

In [12]:
final_log_preds = xgb_model.predict(X_test)
final_dollar_preds = np.exp(final_log_preds) * X_test['floor_area_sqm'] * CURRENT_RPI

mae = mean_absolute_error(y_test_actual, final_dollar_preds)
mape = mean_absolute_percentage_error(y_test_actual, final_dollar_preds) * 100

print(f"RPI Model MAE:  ${mae:,.2f}")
print(f"RPI Model MAPE: {mape:.2f}%")

RPI Model MAE:  $25,959.75
RPI Model MAPE: 3.93%


## 6. Prediction Function

In [13]:
LISTING_PREMIUM = 1.10

def predict_price(town, flat_type, floor_area, sold_year):
    
    town_data = hdb_df[(hdb_df['town'] == town) & 
                       (hdb_df['flat_type'] == flat_type) &
                       (hdb_df['sold_year'] >= 2022)]
    if town_data.empty:
        town_data = hdb_df[(hdb_df['town'] == town) & (hdb_df['flat_type'] == flat_type)]
    if town_data.empty:
        town_data = hdb_df[hdb_df['town'] == town]

    default_lease_commence = int(town_data['lease_commence_date'].median())
    flat_age = sold_year - default_lease_commence
    remaining_lease = 99 - flat_age

    input_dict = {
        'town':                town,
        'flat_type':           flat_type,
        'storey_range':        town_data['storey_range'].mode()[0],
        'floor_area_sqm':      floor_area,
        'flat_model':          town_data['flat_model'].mode()[0],
        'lease_commence_date': default_lease_commence,
        'remaining_lease':     remaining_lease,
        'sold_year':           sold_year,
        'max_floor_lvl':       town_data['max_floor_lvl'].median(),
        'storey_mid':          town_data['storey_mid'].median(),
        'storey_category':     town_data['storey_category'].mode()[0],
        'is_mature_estate':    int(town_data['is_mature_estate'].mode()[0]),
        'flat_age_at_sale':    flat_age,
    }

    for col in TREE_FEATURES:
        if col not in input_dict:
            input_dict[col] = X_train[col].median()

    input_df = pd.DataFrame([input_dict])[TREE_FEATURES]

    for col in categorical_cols:
        if col in input_df.columns:
            try:
                input_df[col] = encoders[col].transform(input_df[col].astype(str))
            except ValueError:
                input_df[col] = X_train[col].mode()[0]

    log_pred = xgb_model.predict(input_df)[0]
    transacted = np.exp(log_pred) * floor_area * CURRENT_RPI
    return transacted * LISTING_PREMIUM

## Example Predictions

In [14]:
price = predict_price('QUEENSTOWN', '4 ROOM', 83, 2026)
print(f"Estimated Transacted Price: ${price:,.2f}")

Estimated Transacted Price: $973,003.94


In [27]:
price = predict_price('QUEENSTOWN', '5 ROOM', 62, 2026)
print(f"Estimated Transacted Price: ${price:,.2f}")

Estimated Transacted Price: $686,771.56
